# Phase 1: Auditable Incident Deduplication

This notebook is the analyst-facing view of the Phase 1 v2 pipeline. It preserves every source report and consolidates entries only when all 11 material fields match exactly or after documented, meaning-preserving normalization: Crossing ID, City, State, Street, County, Railroad, Date/Time, Duration, Reason, Immediate Impacts, and Additional Comments. Temporal proximity remains review-only evidence.

`Date/Time` is interpreted as UTC under the approved source contract. Crossing-local timestamps are derived from the current Form 71 coordinate and historical IANA daylight-saving rules; they do not prove physical-event timing or historical crossing location.

## Required local input files

Download these three untracked source files into `data/` before running the notebook. Their filenames must match exactly; the notebook does not discover or rename alternatives:

- `blocked_crossings_2020through2025.xlsx`
- `blocked_crossings_2025.xlsx`
- `Crossing_Inventory_Data_(Form_71)_-_Current_20260707.csv`

In [ ]:
from pathlib import Path
import importlib
import json
import subprocess
import sys

import pandas as pd

def find_repo_root(start: Path) -> Path:
    """Find the repository by its project layout, not its hidden Git metadata."""
    for candidate in [start.resolve(), *start.resolve().parents]:
        if (candidate / 'analysis').is_dir() and (candidate / 'data').is_dir():
            return candidate
    raise RuntimeError(
        f'Could not find the project root from {start}. Open the notebook from the '
        'Blocked-Crossing-Prediction repository or set repo_root explicitly.'
    )

repo_root = find_repo_root(Path.cwd())
analysis_dir = repo_root / 'analysis'
if str(analysis_dir) not in sys.path:
    sys.path.insert(0, str(analysis_dir))

import incident_deduplication as dedup
dedup = importlib.reload(dedup)
require_input_files = dedup.require_input_files
run_phase_1 = dedup.run_phase_1

# Required untracked files in data/. Filenames are explicit by design.
INPUT_FILENAMES = {
    'authoritative': 'blocked_crossings_2020through2025.xlsx',
    'reconciliation': 'blocked_crossings_2025.xlsx',
    'form_71_inventory': 'Crossing_Inventory_Data_(Form_71)_-_Current_20260707.csv',
}
input_paths = {name: repo_root / 'data' / filename for name, filename in INPUT_FILENAMES.items()}
require_input_files(input_paths)

authoritative_path = input_paths['authoritative']
reconciliation_path = input_paths['reconciliation']
inventory_path = input_paths['form_71_inventory']
config_path = analysis_dir / 'incident_deduplication_config.json'
output_dir = repo_root / 'analysis_outputs' / 'deduplication' / 'v2'
repeat_output_dir = repo_root / 'analysis_outputs' / 'deduplication' / 'v2_repeat'
review_dir = repo_root / 'analysis_outputs' / 'deduplication' / 'review'
review_labels_path = review_dir / 'candidate_review_labels.csv'
acceptance_workflow_stage = 'setup'

## Acceptance preflight

Run the complete automated suite before the long real-data validation. A failing suite stops this notebook before any pipeline outputs are regenerated.

In [ ]:
assert acceptance_workflow_stage == 'setup', 'Restart the kernel and run the notebook from the beginning.'
test_command = ['uv', 'run', 'python', '-m', 'unittest', 'discover', '-s', 'unit_tests', '-p', 'test_*.py']
test_run = subprocess.run(test_command, cwd=repo_root, capture_output=True, text=True)
print(test_run.stdout)
if test_run.stderr:
    print(test_run.stderr, file=sys.stderr)
tests_passed = test_run.returncode == 0
assert tests_passed, f'Automated tests failed with exit code {test_run.returncode}.'
acceptance_workflow_stage = 'tests_passed'

## Two-run real-data validation

Hash the raw inputs, execute the same pipeline into two ignored output directories, compare deterministic artifacts by logical content, and confirm the inputs were not modified. Execution timestamps, durations, and output-directory paths are explicitly excluded from equality.

In [ ]:
assert acceptance_workflow_stage == 'tests_passed', 'Run the acceptance preflight cell first.'
input_hashes_before = dedup.hash_input_files(input_paths)
result = run_phase_1(
    authoritative_path, reconciliation_path, inventory_path, config_path, output_dir
)
repeat_result = run_phase_1(
    authoritative_path, reconciliation_path, inventory_path, config_path, repeat_output_dir
)
input_hashes_after = dedup.hash_input_files(input_paths)
raw_inputs_unchanged = input_hashes_before == input_hashes_after
repeatability = dedup.compare_phase_1_outputs(output_dir, repeat_output_dir)
print(json.dumps(result.summary, indent=2))
assert raw_inputs_unchanged, 'A raw input changed during Phase 1 execution.'
assert repeatability['passed'], json.dumps(repeatability, indent=2)
assert all(result.validations.values()), json.dumps(result.validations, indent=2)
acceptance_workflow_stage = 'real_data_validated'

## Inventory, normalization, and provenance

Unknown duration values remain unmapped; invalid crossing IDs and timestamps remain in the source table and receive documented exceptions rather than canonical incidents.

In [ ]:
inventory_profile = json.loads((output_dir / 'inventory_profile.json').read_text(encoding='utf-8'))
pd.DataFrame([inventory_profile['duration_normalization'], inventory_profile['crossing_id_status']], index=['duration status', 'crossing ID status']).T.fillna(0)

## UTC and crossing-local time

UTC remains the canonical comparison timestamp. Rows without a coordinate-derived IANA zone remain UTC-only and are flagged; no state-level fallback is used.

In [ ]:
timezone_coverage = pd.read_csv(output_dir / 'timezone_assignment_diagnostics.csv')
local_time_diagnostics = pd.read_csv(output_dir / 'local_time_diagnostics.csv')
display(timezone_coverage)
local_time_diagnostics.sort_values(['iana_time_zone', 'reported_local_hour']).head(30)

In [ ]:
source_reports = pd.read_parquet(output_dir / 'source_reports_with_ids.parquet')
source_reports.loc[source_reports['timezone_assignment_status'].eq('assigned'), [
    'source_row_id', 'norm_crossing_id', 'reported_at_utc', 'reported_at_local',
    'iana_time_zone', 'utc_offset_minutes', 'State', 'City'
]].head(20)

## Conservative consolidation and review-only temporal candidates

Matching only Crossing ID, Date/Time, Duration, City, and State does not authorize automatic consolidation. A difference in Street, County, Railroad, Reason, Immediate Impacts, or Additional Comments keeps the entries distinct. Temporal proximity does not alter canonical assignments.

Manual labels are maintained separately from regenerated outputs. They are diagnostic only and never become an automatic merge rule during this remediation.

In [ ]:
assert acceptance_workflow_stage == 'real_data_validated', 'Complete both real-data runs first.'
deduplication_summary = json.loads((output_dir / 'deduplication_summary.json').read_text(encoding='utf-8'))
display(pd.Series(deduplication_summary))
review_sample = pd.read_csv(output_dir / 'candidate_review_sample.csv')
review_dir.mkdir(parents=True, exist_ok=True)
if not review_labels_path.exists():
    dedup.review_label_template(review_sample).to_csv(review_labels_path, index=False)
    print(f'Created review template: {review_labels_path}')
review_labels = pd.read_csv(review_labels_path, keep_default_na=False)
labeled_review_sample, review_summary = dedup.validate_review_labels(review_sample, review_labels)
review_summary['labels_sha256'] = dedup.file_sha256(review_labels_path)
labeled_review_sample.to_csv(output_dir / 'candidate_review_sample_labeled.csv', index=False)
(output_dir / 'candidate_review_summary.json').write_text(json.dumps(review_summary, indent=2, default=str), encoding='utf-8')
display(pd.Series(review_summary))
acceptance_workflow_stage = 'review_evaluated'
labeled_review_sample.head(20)

## Reconciliation, diagnostics, and gate

The 2025 comparison is a normalized full-row multiset comparison, including duplicate multiplicity. A no-report interval must not be called unblocked.

In [ ]:
assert acceptance_workflow_stage == 'review_evaluated', 'Evaluate the deterministic review sample first.'
reconciliation_summary = json.loads((output_dir / 'reconciliation_summary.json').read_text(encoding='utf-8'))
gate_report = json.loads((output_dir / 'phase_1_gate_report.json').read_text(encoding='utf-8'))
display(pd.Series(reconciliation_summary))
display(pd.read_csv(output_dir / 'timestamp_granularity_by_year.csv'))
display(pd.read_csv(output_dir / 'diagnostics_by_year.csv'))
acceptance_workflow_stage = 'diagnostics_reviewed'
gate_report

## Final acceptance decision

The first fresh-kernel run generates the review-label template and repeatability evidence, then correctly ends at `awaiting_review`. Complete and summarize every review label outside the notebook. After the technical checks and review are complete, update the roadmap and perform one final fresh-kernel run from the beginning. Save the notebook after this cell finishes so its execution state is retained.

In [ ]:
notebook_cells_executed_in_order = acceptance_workflow_stage == 'diagnostics_reviewed'
roadmap_text = (repo_root / 'docs' / 'modeling-roadmap.md').read_text(encoding='utf-8')
roadmap_updated = '| 1. Source audit and incident deduplication | Complete' in roadmap_text
claim_text = ' '.join(gate_report['claim_boundary']).lower()
required_artifact_names = [
    'source_reports_with_ids.parquet', 'reported_incidents.parquet',
    'report_incident_crosswalk.parquet', 'documented_exceptions.parquet',
    'duplicate_candidates.parquet', 'candidate_review_sample.csv',
    'inventory_profile.json', 'deduplication_summary.json',
    'diagnostics_by_year.csv', 'diagnostics_by_state.csv',
    'diagnostics_by_crossing.csv', 'diagnostics_by_reason.csv',
    'diagnostics_by_consolidation_tier.csv', 'diagnostics_by_duration_status.csv',
    'diagnostics_by_crossing_volume_tier.csv', 'timestamp_granularity_by_year.csv',
    'crossing_timezones.parquet', 'timezone_assignment_diagnostics.csv',
    'local_time_diagnostics.csv', 'reconciliation_summary.json',
    'reconciliation_discrepancies.parquet', 'candidate_review_summary.json',
    'phase_1_gate_report.json', 'run_manifest.json',
]
required_artifacts_present = all((output_dir / name).is_file() for name in required_artifact_names)
acceptance_checks = {
    'automated_tests_pass': tests_passed,
    'two_real_data_runs_match': repeatability['passed'],
    'raw_inputs_unchanged': raw_inputs_unchanged,
    'source_rows_map_once': result.validations['source_rows_map_once'],
    'unsupported_durations_not_coerced': result.validations['unsupported_duration_bounds_are_null'],
    'all_11_material_fields_required': result.validations['all_incident_groups_share_normalized_11_field_signature'],
    'temporal_candidates_are_review_only': result.validations['candidate_pairs_reference_canonical_incidents'],
    'review_sample_labeled': review_summary['complete'],
    'reconciliation_is_normalized_full_row': reconciliation_summary['comparison'].startswith('normalized full-row multiset'),
    'required_diagnostics_complete': result.validations['diagnostics_have_required_counts'],
    'required_artifacts_present': required_artifacts_present,
    'notebook_cells_executed_in_order': notebook_cells_executed_in_order,
    'claim_language_is_scoped': 'not a verified incident endpoint' in claim_text and 'not unblocked' in claim_text,
    'roadmap_updated': roadmap_updated,
}
acceptance_evidence = {
    'test_command': ' '.join(test_command),
    'input_hashes_before': input_hashes_before,
    'input_hashes_after': input_hashes_after,
    'repeatability': repeatability,
    'review_summary': review_summary,
}
acceptance_report = dedup.write_acceptance_report(output_dir, acceptance_checks, acceptance_evidence)
assert (output_dir / 'phase_1_acceptance_report.json').is_file()
acceptance_workflow_stage = 'acceptance_reported'
display(pd.DataFrame({'criterion': acceptance_checks.keys(), 'passed': acceptance_checks.values()}))
acceptance_report